This notebook runs the model on the preprocessed data. The goal is to predict if the patient will survive to its stay.

In [1]:
import sys
sys.path.append('../')
import pandas as pd

# Reload data

In [2]:
labs = pd.read_csv('data/labs_1_day.csv', index_col = [0, 1], header = [0, 1])
outcomes = pd.read_csv('data/outcomes_1_day.csv', index_col = 0)

In [3]:
outcomes['Death'] = outcomes['Death'] < 8

In [4]:
groups = outcomes[['ETHNICITY', 'GENDER', 'INSURANCE']].copy()
groups.ETHNICITY = outcomes.ETHNICITY.str.contains('BLACK')
groups.GENDER = (outcomes.GENDER == 'M')
groups.INSURANCE = (outcomes.INSURANCE == 'Private')

# Split 

In [5]:
# Results path
results = 'results/classification' 

In [6]:
training = pd.Series(outcomes.index.isin(outcomes.sample(frac = 0.8, random_state = 0).index), index = outcomes.index)

In [7]:
print('Total patients: {}'.format(len(training)))
print('Training patients: {}'.format(training.sum()))

Total patients: 36296
Training patients: 29037


# Imputation

In [8]:
def obtain_membership(groups):
    # Compute the group of different points (if multi group assign an int)
    if isinstance(groups, pd.Series): 
        return groups
    else:
        replace = {g: i for i, g in enumerate(groups.value_counts().index)}
        return groups.apply(lambda x: replace[tuple(x.values.tolist())], axis = 1)

In [9]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
import numpy as np

def imputation(train_index, data, groups, strategy = 'Median', add_missingness = False, max_iter = 10):
    data = data.add_suffix('_data').groupby('Patient').ffill().groupby('Patient').last() # Latest test,
    imputed = data.copy()

    # Data to use to learn imputation
    train_data = imputed.loc[imputed.index.get_level_values('Patient').isin(train_index)]
    train_index = train_data.index
    
    # Compute fill value
    impute = train_data.mean()
    if strategy == 'Mean':
        imputed = imputed.transform(lambda x: x.fillna(impute[x.name]))

    if strategy == 'Group Mean':
        mean_group = train_data.groupby(obtain_membership(groups.loc[train_index])).mean()
        assignement = obtain_membership(groups)
        imputed = imputed.groupby(assignement).transform(lambda x: x.fillna(mean_group.loc[assignement.loc[x.index[0]]][x.name]))
        imputed = imputed.transform(lambda x: x.fillna(impute[x.name])) # Replace any missing data

    if 'MICE' in strategy:
        if 'Group' in strategy:
            # Add group befoer splitting only for imputation
            group_cols = groups.to_frame() if isinstance(groups, pd.Series) else groups.copy()
            group_cols.columns = pd.MultiIndex.from_product([['Group'], group_cols.columns])
            imputed = imputed.join(group_cols)
            train_data = imputed.loc[train_index]
        # MICE Algorithm
        ## 1. Init with median imputation
        missing = imputed.isna()
        imputed = pd.DataFrame(SimpleImputer(strategy = "mean").fit(train_data.values).transform(imputed.values), index = imputed.index, columns = imputed.columns)

        ## 2. Iterate through columns
        ### Find columns with random values (start with the one with least)
        to_impute = missing.sum().sort_values()
        to_impute = to_impute[to_impute > 0]

        ### Impute one by one with regression until convergence
        for _ in range(max_iter):
            for c in to_impute.index:
                #### Take train points for which c is observed to train model
                train_data = imputed.loc[train_index][~missing.loc[train_index][c]]

                #### Fit regression
                lr = LinearRegression().fit(train_data.loc[:, imputed.columns != c].values, train_data[c].values)
                residuals = np.abs(lr.predict(train_data.loc[:, imputed.columns != c].values) - train_data[c])

                #### Draw with normal error
                prev = imputed.copy()
                imputed.loc[missing[c], c] = lr.predict(imputed.loc[:, imputed.columns != c][missing[c]].values) + np.random.normal(scale = np.std(residuals), size = missing[c].sum())
        if 'Group' in strategy:
            # Remove the group columns of imputed data
            imputed = imputed.iloc[:, :-group_cols.shape[1]]

    if add_missingness:
        imputed = imputed.join(data.isna().add_suffix('_missing'))

    return imputed

In [10]:
from utils import Experiment

In [11]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import ParameterGrid

hyperparams = {
    'penalty': ['l2'],
    'C': [0.01, 0.1, 1., 10],
    'solver': ['sag'], 
    'max_iter': [1000],
    'n_jobs': [-1]
}
grid_size = len(ParameterGrid(hyperparams)) # Try each combination once

# sag often reaches max_iter on this data: silence the per-fit warning (see regression_notes.md)
warnings.filterwarnings('ignore', category = ConvergenceWarning)

In [12]:
imputations = {
                'Mean': {'strategy': 'Mean'},
                'Mean Missing': {'strategy': 'Mean', 'add_missingness': True},

                'Group Ethnicity Mean': {'strategy': 'Group Mean', 'group': groups.ETHNICITY},
                'Group Sex Mean': {'strategy': 'Group Mean', 'group': groups.GENDER},
                'Group Insurance Mean': {'strategy': 'Group Mean', 'group': groups.INSURANCE},
                'Group All Mean': {'strategy': 'Group Mean', 'group': groups},

                'Group Ethnicity Mean Missing': {'strategy': 'Group Mean', 'group': groups.ETHNICITY, 'add_missingness': True},
                'Group Sex Mean Missing': {'strategy': 'Group Mean', 'group': groups.GENDER, 'add_missingness': True},
                'Group Insurance Mean Missing': {'strategy': 'Group Mean', 'group': groups.INSURANCE, 'add_missingness': True},
                'Group All Mean Missing': {'strategy': 'Group Mean', 'group': groups, 'add_missingness': True},

                'MICE': {'strategy': 'MICE', 'n_iter': 10},
                'MICE Missing': {'strategy': 'MICE', 'n_iter': 10, 'add_missingness': True},

                'Group Ethnicity MICE': {'strategy': 'Group MICE', 'n_iter': 10, 'group': groups.ETHNICITY},
                'Group Sex MICE': {'strategy': 'Group MICE', 'n_iter': 10, 'group': groups.GENDER},
                'Group Insurance MICE': {'strategy': 'Group MICE', 'n_iter': 10, 'group': groups.INSURANCE},
                'Group All MICE': {'strategy': 'Group MICE', 'n_iter': 10, 'group': groups},

                'Group Ethnicity MICE Missing': {'strategy': 'Group MICE', 'n_iter': 10, 'group': groups.ETHNICITY, 'add_missingness': True},
                'Group Sex MICE Missing': {'strategy': 'Group MICE', 'n_iter': 10, 'group': groups.GENDER, 'add_missingness': True},
                'Group Insurance MICE Missing': {'strategy': 'Group MICE', 'n_iter': 10, 'group': groups.INSURANCE, 'add_missingness': True},
                'Group All MICE Missing': {'strategy': 'Group MICE', 'n_iter': 10, 'group': groups,'add_missingness': True}
              }

In [13]:
for name, params in imputations.items():
    print('Imputation strategy: ', name)
    n_iter = params.pop('n_iter', 1)
    subgroup = params.pop('group', groups)

    predictions = []
    for iter in range(n_iter):
        last = imputation(training[training].index, labs, subgroup, **params)
        assert (last == -1).sum().sum() == 0, "Non imputed values"

        se = Experiment.create(model = 'log', hyper_grid = hyperparams, n_iter = grid_size, save = False, path = results + name)
        pred = se.train(last, outcomes.Death, training)
        if pred is None: break # Reload previous copy
        predictions.append(pred)
    else:
        # Average Multiple imputations models
        used = [p.Use for p in predictions][-1]
        predictions = pd.concat([p[1] for p in predictions], axis = 1)
        predictions = pd.concat({'Mean': predictions.mean(1), 'Std': predictions.std(1)}, axis = 1)
        se = Experiment.create(model = 'log', hyper_grid = hyperparams, n_iter = grid_size, path = results + name)
        se.save_results(predictions, used)

Imputation strategy:  Mean
results/classificationMean
results/classificationMean
Imputation strategy:  Mean Missing
results/classificationMean Missing
results/classificationMean Missing
Imputation strategy:  Group Ethnicity Mean
results/classificationGroup Ethnicity Mean
results/classificationGroup Ethnicity Mean
Imputation strategy:  Group Sex Mean
results/classificationGroup Sex Mean
results/classificationGroup Sex Mean
Imputation strategy:  Group Insurance Mean
results/classificationGroup Insurance Mean
results/classificationGroup Insurance Mean
Imputation strategy:  Group All Mean


/tmp/ipykernel_73997/4275168448.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  imputed = imputed.groupby(assignement).transform(lambda x: x.fillna(mean_group.loc[assignement.loc[x.index[0]]][x.name]))


results/classificationGroup All Mean
results/classificationGroup All Mean
Imputation strategy:  Group Ethnicity Mean Missing
results/classificationGroup Ethnicity Mean Missing
results/classificationGroup Ethnicity Mean Missing
Imputation strategy:  Group Sex Mean Missing
results/classificationGroup Sex Mean Missing
results/classificationGroup Sex Mean Missing
Imputation strategy:  Group Insurance Mean Missing
results/classificationGroup Insurance Mean Missing
results/classificationGroup Insurance Mean Missing
Imputation strategy:  Group All Mean Missing


/tmp/ipykernel_73997/4275168448.py:21: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  imputed = imputed.groupby(assignement).transform(lambda x: x.fillna(mean_group.loc[assignement.loc[x.index[0]]][x.name]))


results/classificationGroup All Mean Missing
results/classificationGroup All Mean Missing
Imputation strategy:  MICE
results/classificationMICE
results/classificationMICE
results/classificationMICE
results/classificationMICE
results/classificationMICE
results/classificationMICE
results/classificationMICE
results/classificationMICE
results/classificationMICE
results/classificationMICE
results/classificationMICE
Imputation strategy:  MICE Missing
results/classificationMICE Missing
results/classificationMICE Missing
results/classificationMICE Missing
results/classificationMICE Missing
results/classificationMICE Missing
results/classificationMICE Missing
results/classificationMICE Missing
results/classificationMICE Missing
results/classificationMICE Missing
results/classificationMICE Missing
results/classificationMICE Missing
Imputation strategy:  Group Ethnicity MICE
results/classificationGroup Ethnicity MICE
results/classificationGroup Ethnicity MICE
results/classificationGroup Ethnicity